In [ ]:
from pathlib import Path
import json, math, random, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from torchvision import transforms
from torchvision.datasets import OxfordIIITPet
from torchvision.models import mobilenet_v2, MobileNet_V2_Weights
from torch.utils.data import DataLoader, Subset, TensorDataset
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report

SEED=24110085
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE=torch.device("mps" if hasattr(torch.backends,"mps") and torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu"))
DATA_ROOT=Path("oxford_pet_data")
OUT=Path("experiment5_outputs")
OUT.mkdir(exist_ok=True)
NUM_CLASSES=37
IMAGE_SIZE=224
PROBE_EPOCHS=8
CV_EPOCHS=8
FINE_TUNE_EPOCHS=5
PARTIAL_CV_EPOCHS=2
IMAGENET_MEAN=[0.485,0.456,0.406]
IMAGENET_STD=[0.229,0.224,0.225]
transform=transforms.Compose([transforms.Resize((IMAGE_SIZE,IMAGE_SIZE)),transforms.ToTensor(),transforms.Normalize(IMAGENET_MEAN,IMAGENET_STD)])
trainval=OxfordIIITPet(DATA_ROOT,split="trainval",target_types="category",transform=transform,download=True)
testset=OxfordIIITPet(DATA_ROOT,split="test",target_types="category",transform=transform,download=True)
class_names=list(trainval.classes)
all_targets=np.array(getattr(trainval,"_labels",[trainval[i][1] for i in range(len(trainval))]),dtype=np.int64)
train_idx,val_idx=train_test_split(np.arange(len(trainval)),test_size=0.20,stratify=all_targets,random_state=SEED)
weights=MobileNet_V2_Weights.DEFAULT
base=mobilenet_v2(weights=weights)
base.classifier=nn.Identity()
base=base.to(DEVICE).eval()
for p in base.parameters():
    p.requires_grad=False

def cache_features(dataset,indices=None,batch_size=64):
    ds=dataset if indices is None else Subset(dataset,indices)
    loader=DataLoader(ds,batch_size=batch_size,shuffle=False,num_workers=0,pin_memory=torch.cuda.is_available())
    feats=[]
    ys=[]
    with torch.no_grad():
        for x,y in loader:
            feats.append(base(x.to(DEVICE)).cpu())
            ys.append(torch.as_tensor(y).long().cpu())
    return torch.cat(feats),torch.cat(ys)

X_train,y_train=cache_features(trainval,train_idx)
X_val,y_val=cache_features(trainval,val_idx)
X_full,y_full=cache_features(trainval)
X_test,y_test=cache_features(testset)
torch.save({"X_train":X_train,"y_train":y_train,"X_val":X_val,"y_val":y_val,"X_full":X_full,"y_full":y_full,"X_test":X_test,"y_test":y_test,"class_names":class_names},OUT/"mobilenetv2_features.pt")

fig,axes=plt.subplots(4,10,figsize=(15,6))
seen=set()
for i in range(len(trainval)):
    img,label=trainval[i]
    if int(label) not in seen:
        r=len(seen)//10
        c=len(seen)%10
        z=img.permute(1,2,0).numpy()*np.array(IMAGENET_STD)+np.array(IMAGENET_MEAN)
        axes[r,c].imshow(np.clip(z,0,1))
        axes[r,c].set_title(class_names[int(label)].replace("_"," "),fontsize=6)
        axes[r,c].axis("off")
        seen.add(int(label))
        if len(seen)==NUM_CLASSES:
            break
for j in range(NUM_CLASSES,40):
    axes[j//10,j%10].axis("off")
fig.suptitle("Oxford-IIIT Pet: One Sample from Each Breed")
fig.tight_layout()
fig.savefig(OUT/"sample_breeds.png",dpi=180,bbox_inches="tight")
plt.close(fig)


In [ ]:
class FeatureHead(nn.Module):
    def __init__(self,dropout=0.0,batch_norm=False):
        super().__init__()
        layers=[nn.Linear(1280,128)]
        if batch_norm:
            layers.append(nn.BatchNorm1d(128))
        layers.extend([nn.ReLU(),nn.Dropout(dropout),nn.Linear(128,NUM_CLASSES)])
        self.net=nn.Sequential(*layers)
    def forward(self,x):
        return self.net(x)

def initialize_model(model,kind):
    for layer in model.modules():
        if isinstance(layer,nn.Linear):
            if kind=="zero":
                nn.init.zeros_(layer.weight)
            elif kind=="random":
                nn.init.normal_(layer.weight,0.0,0.05)
            elif kind=="xavier":
                nn.init.xavier_uniform_(layer.weight)
            elif kind=="he":
                nn.init.kaiming_normal_(layer.weight,nonlinearity="relu")
            if layer.bias is not None:
                nn.init.zeros_(layer.bias)

def make_optimizer(name,params,lr,weight_decay=0.0):
    if name=="SGD":
        return optim.SGD(params,lr=lr,weight_decay=weight_decay)
    if name=="Momentum":
        return optim.SGD(params,lr=lr,momentum=0.9,weight_decay=weight_decay)
    if name=="RMSProp":
        return optim.RMSprop(params,lr=lr,weight_decay=weight_decay)
    return optim.Adam(params,lr=lr,weight_decay=weight_decay)

def tensor_loader(X,y,batch_size,shuffle):
    return DataLoader(TensorDataset(X,y),batch_size=batch_size,shuffle=shuffle,num_workers=0)

def run_epoch(model,loader,criterion,optimizer=None):
    model.train(optimizer is not None)
    loss_sum=0.0
    correct=0
    count=0
    for x,y in loader:
        x=x.to(DEVICE)
        y=y.to(DEVICE)
        if optimizer is not None:
            optimizer.zero_grad(set_to_none=True)
        logits=model(x)
        loss=criterion(logits,y)
        if optimizer is not None:
            loss.backward()
            optimizer.step()
        loss_sum+=float(loss.item())*len(y)
        correct+=int((logits.argmax(1)==y).sum().item())
        count+=len(y)
    return loss_sum/count,correct/count

def run_image_epoch(model,loader,criterion,optimizer=None,lock_feature_bn=False):
    model.train(optimizer is not None)
    if optimizer is not None and lock_feature_bn:
        for layer in model.features.modules():
            if isinstance(layer,nn.BatchNorm2d):
                layer.eval()
    loss_sum=0.0
    correct=0
    count=0
    for x,y in loader:
        x=x.to(DEVICE)
        y=y.to(DEVICE)
        if optimizer is not None:
            optimizer.zero_grad(set_to_none=True)
        logits=model(x)
        loss=criterion(logits,y)
        if optimizer is not None:
            loss.backward()
            optimizer.step()
        loss_sum+=float(loss.item())*len(y)
        correct+=int((logits.argmax(1)==y).sum().item())
        count+=len(y)
    return loss_sum/count,correct/count

def fit_head(Xtr,ytr,Xva,yva,initializer="he",optimizer_name="Adam",lr=1e-3,batch_size=32,dropout=0.0,batch_norm=False,l2=0.0,epochs=PROBE_EPOCHS):
    torch.manual_seed(SEED)
    model=FeatureHead(dropout,batch_norm).to(DEVICE)
    initialize_model(model,initializer)
    optimizer=make_optimizer(optimizer_name,model.parameters(),lr,l2)
    criterion=nn.CrossEntropyLoss()
    tr=tensor_loader(Xtr,ytr,batch_size,True)
    va=tensor_loader(Xva,yva,max(batch_size,64),False)
    h={"train_loss":[],"train_acc":[],"val_loss":[],"val_acc":[],"seconds":[]}
    t0=time.perf_counter()
    for _ in range(epochs):
        e0=time.perf_counter()
        a,b=run_epoch(model,tr,criterion,optimizer)
        c,d=run_epoch(model,va,criterion,None)
        h["train_loss"].append(a)
        h["train_acc"].append(b)
        h["val_loss"].append(c)
        h["val_acc"].append(d)
        h["seconds"].append(time.perf_counter()-e0)
    h["total_time"]=time.perf_counter()-t0
    return model,h

def predict_head(model,X,y,batch_size=128):
    loader=tensor_loader(X,y,batch_size,False)
    model.eval()
    truth=[]
    pred=[]
    with torch.no_grad():
        for a,b in loader:
            p=model(a.to(DEVICE)).argmax(1).cpu()
            truth.extend(b.numpy().tolist())
            pred.extend(p.numpy().tolist())
    return np.array(truth),np.array(pred)

def weighted_metrics(y,p):
    pr,re,f1,_=precision_recall_fscore_support(y,p,average="weighted",zero_division=0)
    return {"accuracy":float(accuracy_score(y,p)),"precision":float(pr),"recall":float(re),"f1":float(f1)}

def converge_epoch(hist):
    v=np.array(hist["val_acc"])
    target=0.99*v.max()
    return int(np.argmax(v>=target)+1)

def save_lines(data,key,title,ylabel,path,percent=False):
    fig,ax=plt.subplots(figsize=(8,4.8))
    for name,h in data.items():
        vals=np.array(h[key])*(100 if percent else 1)
        ax.plot(np.arange(1,len(vals)+1),vals,marker="o",markersize=3,label=name)
    ax.set_xlabel("Epoch")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(alpha=.25)
    ax.legend()
    fig.tight_layout()
    fig.savefig(path,dpi=180,bbox_inches="tight")
    plt.close(fig)

def save_train_val(data,title,ykey,path,percent=False):
    fig,ax=plt.subplots(figsize=(9,5.2))
    for name,h in data.items():
        scale=100 if percent else 1
        ax.plot(np.arange(1,len(h["train_"+ykey])+1),np.array(h["train_"+ykey])*scale,label=f"{name} Train")
        ax.plot(np.arange(1,len(h["val_"+ykey])+1),np.array(h["val_"+ykey])*scale,linestyle="--",label=f"{name} Val")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Accuracy (%)" if ykey=="acc" else "Loss")
    ax.set_title(title)
    ax.grid(alpha=.25)
    ax.legend(ncol=2,fontsize=8)
    fig.tight_layout()
    fig.savefig(path,dpi=180,bbox_inches="tight")
    plt.close(fig)

bn_x=np.array([2.,4.,6.,8.])
bn_mean=float(bn_x.mean())
bn_var=float(((bn_x-bn_mean)**2).mean())
bn_std=float(math.sqrt(bn_var))
bn_normalized=(bn_x-bn_mean)/bn_std
conv_examples=pd.DataFrame([[224,3,1,0,222],[224,3,1,1,224],[224,3,2,0,111],[224,3,2,1,112],[224,5,1,0,220]],columns=["N","K","S","P","Output"])


In [ ]:
initialization={}
init_models={}
for kind in ["zero","random","xavier","he"]:
    m,h=fit_head(X_train,y_train,X_val,y_val,initializer=kind,dropout=0.0,batch_norm=False)
    initialization[kind]=h
    init_models[kind]=m
save_lines(initialization,"train_loss","Training Loss for Weight Initializers","Training Loss",OUT/"plot01_initialization_training_loss.png")
save_lines(initialization,"val_acc","Validation Accuracy for Weight Initializers","Validation Accuracy (%)",OUT/"plot02_initialization_validation_accuracy.png",True)
best_initializer=max(initialization,key=lambda k:max(initialization[k]["val_acc"]))

regularization={}
reg_models={}
reg_specs={"None":(0.0,False,0.0),"L2":(0.0,False,1e-4),"Dropout":(0.5,False,0.0),"Batch Normalization":(0.0,True,0.0)}
for name,(drop,bn,l2) in reg_specs.items():
    m,h=fit_head(X_train,y_train,X_val,y_val,initializer=best_initializer,dropout=drop,batch_norm=bn,l2=l2)
    regularization[name]=h
    reg_models[name]=m
save_train_val(regularization,"Regularization: Training and Validation Accuracy","acc",OUT/"plot03_regularization_accuracy.png",True)
save_train_val(regularization,"Regularization: Training and Validation Loss","loss",OUT/"plot04_regularization_loss.png")
bn_compare={"Without BN":regularization["None"],"With BN":regularization["Batch Normalization"]}
save_lines(bn_compare,"val_acc","With vs Without Batch Normalization","Validation Accuracy (%)",OUT/"plot05_batch_normalization.png",True)
best_regularization=max(regularization,key=lambda k:max(regularization[k]["val_acc"]))

optimizers_result={}
optimizer_models={}
optimizer_rows=[]
for name in ["SGD","Momentum","RMSProp","Adam"]:
    m,h=fit_head(X_train,y_train,X_val,y_val,initializer=best_initializer,optimizer_name=name,lr=1e-3,batch_size=32,dropout=0.0)
    optimizers_result[name]=h
    optimizer_models[name]=m
    optimizer_rows.append([name,h["train_loss"][-1],100*max(h["val_acc"]),converge_epoch(h),h["total_time"]])
optimizer_table=pd.DataFrame(optimizer_rows,columns=["Optimizer","Final Loss","Best Val Accuracy (%)","Epoch to Converge","Time (s)"])
save_lines(optimizers_result,"train_loss","Training Loss for Different Optimizers","Training Loss",OUT/"plot06_optimizer_training_loss.png")
save_lines(optimizers_result,"val_acc","Validation Accuracy for Different Optimizers","Validation Accuracy (%)",OUT/"plot07_optimizer_validation_accuracy.png",True)
best_optimizer=max(optimizers_result,key=lambda k:max(optimizers_result[k]["val_acc"]))

lr_results={}
for lr in [1e-3,1e-4]:
    _,h=fit_head(X_train,y_train,X_val,y_val,initializer=best_initializer,optimizer_name="Adam",lr=lr,batch_size=32,dropout=0.0)
    lr_results[lr]=100*max(h["val_acc"])
bs_results={}
for bs in [16,32,64]:
    _,h=fit_head(X_train,y_train,X_val,y_val,initializer=best_initializer,optimizer_name="Adam",lr=1e-3,batch_size=bs,dropout=0.0)
    bs_results[bs]=100*max(h["val_acc"])
drop_results={}
for d in [0.0,0.25,0.5]:
    _,h=fit_head(X_train,y_train,X_val,y_val,initializer=best_initializer,optimizer_name="Adam",lr=1e-3,batch_size=32,dropout=d)
    drop_results[d]=100*max(h["val_acc"])
for values,title,xlabel,path in [(lr_results,"Learning Rate vs Validation Accuracy","Learning Rate",OUT/"plot08_learning_rate.png"),(bs_results,"Batch Size vs Validation Accuracy","Batch Size",OUT/"plot09_batch_size.png"),(drop_results,"Dropout Rate vs Validation Accuracy","Dropout Rate",OUT/"plot10_dropout_rate.png")]:
    fig,ax=plt.subplots(figsize=(6.8,4.4))
    xs=list(values.keys())
    ys=list(values.values())
    ax.plot(range(len(xs)),ys,marker="o")
    ax.set_xticks(range(len(xs)),[str(x) for x in xs])
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Best Validation Accuracy (%)")
    ax.set_title(title)
    ax.grid(alpha=.25)
    fig.tight_layout()
    fig.savefig(path,dpi=180,bbox_inches="tight")
    plt.close(fig)
best_lr=max(lr_results,key=lr_results.get)
best_batch=max(bs_results,key=bs_results.get)
best_dropout=max(drop_results,key=drop_results.get)


In [ ]:
def image_model(dropout=0.25,batch_norm=False,pretrained=True):
    m=mobilenet_v2(weights=weights if pretrained else None)
    m.classifier=FeatureHead(dropout,batch_norm).net
    return m.to(DEVICE)

def copy_head(head,model):
    model.classifier.load_state_dict(head.net.state_dict())

def image_loaders(train_indices,val_indices,batch_size):
    tr=DataLoader(Subset(trainval,train_indices),batch_size=batch_size,shuffle=True,num_workers=0,pin_memory=torch.cuda.is_available())
    va=DataLoader(Subset(trainval,val_indices),batch_size=batch_size,shuffle=False,num_workers=0,pin_memory=torch.cuda.is_available())
    return tr,va

def fit_image(model,tr,va,lr,epochs,freeze_base=True):
    if freeze_base:
        for p in model.features.parameters():
            p.requires_grad=False
    criterion=nn.CrossEntropyLoss()
    optimizer=optim.Adam(filter(lambda p:p.requires_grad,model.parameters()),lr=lr)
    h={"train_loss":[],"train_acc":[],"val_loss":[],"val_acc":[],"seconds":[]}
    t0=time.perf_counter()
    for _ in range(epochs):
        e0=time.perf_counter()
        a,b=run_image_epoch(model,tr,criterion,optimizer,True)
        c,d=run_image_epoch(model,va,criterion,None,True)
        h["train_loss"].append(a)
        h["train_acc"].append(b)
        h["val_loss"].append(c)
        h["val_acc"].append(d)
        h["seconds"].append(time.perf_counter()-e0)
    h["total_time"]=time.perf_counter()-t0
    return h

def enable_partial(model,last_blocks=4):
    for p in model.features.parameters():
        p.requires_grad=False
    children=list(model.features.children())
    for layer in children[-last_blocks:]:
        for p in layer.parameters():
            p.requires_grad=True
    for layer in model.features.modules():
        if isinstance(layer,nn.BatchNorm2d):
            layer.eval()
            for p in layer.parameters():
                p.requires_grad=False

reg_drop,reg_bn,reg_l2=reg_specs[best_regularization]
feature_config={"initializer":best_initializer,"optimizer":best_optimizer,"lr":float(best_lr),"batch_size":int(best_batch),"dropout":float(best_dropout),"batch_norm":reg_bn,"l2":reg_l2}
feature_head,feature_hist=fit_head(X_train,y_train,X_val,y_val,initializer=feature_config["initializer"],optimizer_name=feature_config["optimizer"],lr=feature_config["lr"],batch_size=feature_config["batch_size"],dropout=feature_config["dropout"],batch_norm=feature_config["batch_norm"],l2=feature_config["l2"],epochs=PROBE_EPOCHS)
tr_img,va_img=image_loaders(train_idx,val_idx,feature_config["batch_size"])
fine_tune_lr_histories={}
for fine_lr in [1e-4,1e-5]:
    transfer_model=image_model(feature_config["dropout"],feature_config["batch_norm"],True)
    copy_head(feature_head,transfer_model)
    enable_partial(transfer_model,4)
    fine_tune_lr_histories[fine_lr]=fit_image(transfer_model,tr_img,va_img,fine_lr,FINE_TUNE_EPOCHS,False)
best_fine_tune_lr=max(fine_tune_lr_histories,key=lambda x:max(fine_tune_lr_histories[x]["val_acc"]))
fine_hist=fine_tune_lr_histories[best_fine_tune_lr]

fig,ax=plt.subplots(figsize=(8,4.8))
x1=np.arange(1,PROBE_EPOCHS+1)
x2=np.arange(PROBE_EPOCHS+1,PROBE_EPOCHS+FINE_TUNE_EPOCHS+1)
ax.plot(x1,np.array(feature_hist["val_acc"])*100,marker="o",label="Feature Extraction")
ax.plot(x2,np.array(fine_hist["val_acc"])*100,marker="o",label=f"Fine-Tuning LR={best_fine_tune_lr:g}")
ax.axvline(PROBE_EPOCHS+.5,linestyle="--",alpha=.6)
ax.set_xlabel("Epoch")
ax.set_ylabel("Validation Accuracy (%)")
ax.set_title("Feature Extraction vs Fine-Tuning")
ax.grid(alpha=.25)
ax.legend()
fig.tight_layout()
fig.savefig(OUT/"plot11_feature_extraction_vs_finetuning.png",dpi=180,bbox_inches="tight")
plt.close(fig)

fig,ax=plt.subplots(figsize=(8,4.8))
ax.plot(x1,feature_hist["train_loss"],label="Feature Train Loss")
ax.plot(x1,feature_hist["val_loss"],linestyle="--",label="Feature Val Loss")
ax.plot(x2,fine_hist["train_loss"],label="Fine-Tune Train Loss")
ax.plot(x2,fine_hist["val_loss"],linestyle="--",label="Fine-Tune Val Loss")
ax.axvline(PROBE_EPOCHS+.5,linestyle="--",alpha=.6)
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.set_title("Training and Validation Loss Before and After Fine-Tuning")
ax.grid(alpha=.25)
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(OUT/"plot12_transfer_loss.png",dpi=180,bbox_inches="tight")
plt.close(fig)


In [ ]:
def cv_head_config(config,folds=5,epochs=CV_EPOCHS):
    skf=StratifiedKFold(n_splits=folds,shuffle=True,random_state=SEED)
    scores=[]
    times=[]
    y_np=y_full.numpy()
    for tr,va in skf.split(np.zeros(len(y_np)),y_np):
        t0=time.perf_counter()
        _,h=fit_head(X_full[tr],y_full[tr],X_full[va],y_full[va],initializer=config.get("initializer",best_initializer),optimizer_name=config.get("optimizer","Adam"),lr=config.get("lr",1e-3),batch_size=config.get("batch_size",32),dropout=config.get("dropout",0.0),batch_norm=config.get("batch_norm",False),l2=config.get("l2",0.0),epochs=epochs)
        scores.append(float(max(h["val_acc"])))
        times.append(time.perf_counter()-t0)
    return {"folds":scores,"mean":float(np.mean(scores)),"sd":float(np.std(scores,ddof=0)),"time_s":float(sum(times))}

def cv_partial_config(config,folds=5,head_epochs=CV_EPOCHS,fine_epochs=PARTIAL_CV_EPOCHS):
    skf=StratifiedKFold(n_splits=folds,shuffle=True,random_state=SEED)
    scores=[]
    times=[]
    y_np=y_full.numpy()
    for tr,va in skf.split(np.zeros(len(y_np)),y_np):
        t0=time.perf_counter()
        head,_=fit_head(X_full[tr],y_full[tr],X_full[va],y_full[va],initializer=config.get("initializer",best_initializer),optimizer_name=config.get("optimizer","Adam"),lr=config.get("head_lr",config.get("lr",1e-3)),batch_size=config.get("batch_size",32),dropout=config.get("dropout",0.0),batch_norm=config.get("batch_norm",False),l2=config.get("l2",0.0),epochs=head_epochs)
        model=image_model(config.get("dropout",0.0),config.get("batch_norm",False),True)
        copy_head(head,model)
        enable_partial(model,config.get("last_blocks",4))
        tr_loader,va_loader=image_loaders(tr,va,config.get("batch_size",32))
        h=fit_image(model,tr_loader,va_loader,config.get("fine_lr",1e-5),fine_epochs,False)
        scores.append(float(max(h["val_acc"])))
        times.append(time.perf_counter()-t0)
        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    return {"folds":scores,"mean":float(np.mean(scores)),"sd":float(np.std(scores,ddof=0)),"time_s":float(sum(times))}

configs={
    "C1 Baseline":{"initializer":"he","optimizer":"Adam","lr":1e-3,"batch_size":32,"dropout":0.0,"batch_norm":False,"l2":0.0},
    "C2 Best Initialization":{"initializer":best_initializer,"optimizer":"Adam","lr":1e-3,"batch_size":32,"dropout":0.0,"batch_norm":False,"l2":0.0},
    "C3 Best Regularization":{"initializer":best_initializer,"optimizer":"Adam","lr":1e-3,"batch_size":32,"dropout":reg_drop,"batch_norm":reg_bn,"l2":reg_l2},
    "C4 Best Hyperparameters":{"initializer":best_initializer,"optimizer":best_optimizer,"lr":float(best_lr),"batch_size":int(best_batch),"dropout":float(best_dropout),"batch_norm":reg_bn,"l2":reg_l2}
}
cv_results={name:cv_head_config(cfg) for name,cfg in configs.items()}
cv_table=pd.DataFrame([[name,*r["folds"],r["mean"],r["sd"],r["time_s"]] for name,r in cv_results.items()],columns=["Configuration","F1","F2","F3","F4","F5","Mean","SD","Time (s)"])
fig,ax=plt.subplots(figsize=(8,4.8))
names=list(cv_results)
means=np.array([cv_results[n]["mean"] for n in names])*100
sds=np.array([cv_results[n]["sd"] for n in names])*100
ax.bar(range(len(names)),means,yerr=sds,capsize=5)
ax.set_xticks(range(len(names)),[x.split()[0] for x in names])
ax.set_xlabel("Hyperparameter Configuration")
ax.set_ylabel("Mean Validation Accuracy (%)")
ax.set_title("5-Fold Cross-Validation Accuracy")
ax.grid(axis="y",alpha=.25)
fig.tight_layout()
fig.savefig(OUT/"plot13_cross_validation.png",dpi=180,bbox_inches="tight")
plt.close(fig)
selected_name=max(cv_results,key=lambda k:cv_results[k]["mean"])
selected_config=configs[selected_name]
fine_cv_config={**selected_config,"head_lr":selected_config["lr"],"fine_lr":float(best_fine_tune_lr),"last_blocks":4}
fine_tuned_cv=cv_partial_config(fine_cv_config)


In [ ]:
def train_full_head(config,epochs=PROBE_EPOCHS):
    return fit_head(X_full,y_full,X_full,y_full,initializer=config.get("initializer",best_initializer),optimizer_name=config.get("optimizer","Adam"),lr=config.get("lr",1e-3),batch_size=config.get("batch_size",32),dropout=config.get("dropout",0.0),batch_norm=config.get("batch_norm",False),l2=config.get("l2",0.0),epochs=epochs)

def eval_config_on_test(config):
    t0=time.perf_counter()
    model,h=train_full_head(config)
    y,p=predict_head(model,X_test,y_test)
    m=weighted_metrics(y,p)
    m["training_time_s"]=time.perf_counter()-t0
    return m,model,h

def train_partial_and_test(config,fine_lr=None,head_epochs=PROBE_EPOCHS,fine_epochs=FINE_TUNE_EPOCHS):
    t0=time.perf_counter()
    head,h=train_full_head(config,head_epochs)
    model=image_model(config.get("dropout",0.0),config.get("batch_norm",False),True)
    copy_head(head,model)
    enable_partial(model,config.get("last_blocks",4))
    full_loader=DataLoader(trainval,batch_size=int(config.get("batch_size",32)),shuffle=True,num_workers=0,pin_memory=torch.cuda.is_available())
    criterion=nn.CrossEntropyLoss()
    optimizer=optim.Adam(filter(lambda p:p.requires_grad,model.parameters()),lr=float(fine_lr if fine_lr is not None else config.get("fine_lr",best_fine_tune_lr)))
    hist={"train_loss":[],"train_acc":[],"seconds":[]}
    for _ in range(fine_epochs):
        e0=time.perf_counter()
        a,b=run_image_epoch(model,full_loader,criterion,optimizer,True)
        hist["train_loss"].append(a)
        hist["train_acc"].append(b)
        hist["seconds"].append(time.perf_counter()-e0)
    test_loader=DataLoader(testset,batch_size=64,shuffle=False,num_workers=0,pin_memory=torch.cuda.is_available())
    truth=[]
    pred=[]
    model.eval()
    with torch.no_grad():
        for x,y in test_loader:
            p=model(x.to(DEVICE)).argmax(1).cpu().numpy()
            truth.extend(np.asarray(y).tolist())
            pred.extend(p.tolist())
    truth=np.array(truth)
    pred=np.array(pred)
    metrics=weighted_metrics(truth,pred)
    metrics["training_time_s"]=time.perf_counter()-t0
    metrics["total_parameters"]=int(sum(p.numel() for p in model.parameters()))
    metrics["trainable_parameters"]=int(sum(p.numel() for p in model.parameters() if p.requires_grad))
    return metrics,model,hist,truth,pred

representatives={
    "Baseline":configs["C1 Baseline"],
    "Best Initialization":configs["C2 Best Initialization"],
    "Best Regularization":configs["C3 Best Regularization"],
    "Best Optimizer":{**configs["C1 Baseline"],"optimizer":best_optimizer},
    "Best Hyperparameters":configs["C4 Best Hyperparameters"]
}
representative_test={}
for name,cfg in representatives.items():
    m,_,_=eval_config_on_test(cfg)
    representative_test[name]=m

final_metrics,final_image,final_hist,truth,pred=train_partial_and_test({**selected_config,"last_blocks":4},float(best_fine_tune_lr))
final_metrics["mean_cv_accuracy"]=fine_tuned_cv["mean"]
final_metrics["cv_sd"]=fine_tuned_cv["sd"]
final_metrics["selected_configuration"]=selected_name
cm=confusion_matrix(truth,pred,labels=np.arange(NUM_CLASSES))
report=classification_report(truth,pred,target_names=class_names,zero_division=0,output_dict=True)
class_correct=np.diag(cm)
best_class_index=int(np.argmax(class_correct))
cm_off=cm.copy()
np.fill_diagonal(cm_off,0)
confused_true,confused_pred=np.unravel_index(np.argmax(cm_off),cm_off.shape)
final_metrics["best_class"]=class_names[best_class_index]
final_metrics["best_class_correct"]=int(class_correct[best_class_index])
final_metrics["most_confused_true"]=class_names[int(confused_true)]
final_metrics["most_confused_pred"]=class_names[int(confused_pred)]
final_metrics["most_confused_count"]=int(cm_off[confused_true,confused_pred])

fig,ax=plt.subplots(figsize=(13,11))
im=ax.imshow(cm,aspect="auto")
ax.set_xticks(np.arange(NUM_CLASSES),[x.replace("_"," ") for x in class_names],rotation=90,fontsize=6)
ax.set_yticks(np.arange(NUM_CLASSES),[x.replace("_"," ") for x in class_names],fontsize=6)
ax.set_xlabel("Predicted Class")
ax.set_ylabel("Actual Class")
ax.set_title("Final Fine-Tuned MobileNetV2 Confusion Matrix")
fig.colorbar(im,ax=ax,fraction=.046,pad=.04)
fig.tight_layout()
fig.savefig(OUT/"plot14_confusion_matrix.png",dpi=180,bbox_inches="tight")
plt.close(fig)

mis=np.where(truth!=pred)[0][:12]
fig,axes=plt.subplots(3,4,figsize=(12,9))
for ax,j in zip(axes.ravel(),mis):
    img,label=testset[int(j)]
    z=img.permute(1,2,0).numpy()*np.array(IMAGENET_STD)+np.array(IMAGENET_MEAN)
    ax.imshow(np.clip(z,0,1))
    ax.set_title(f"T: {class_names[int(label)].replace('_',' ')}\nP: {class_names[int(pred[j])].replace('_',' ')}",fontsize=8)
    ax.axis("off")
for ax in axes.ravel()[len(mis):]:
    ax.axis("off")
fig.suptitle("Representative Misclassified Images")
fig.tight_layout()
fig.savefig(OUT/"plot15_misclassified_images.png",dpi=180,bbox_inches="tight")
plt.close(fig)

additional={
    "A1":{"initializer":best_initializer,"optimizer":"Adam","lr":1e-4,"batch_size":16,"dropout":0.25,"batch_norm":False,"l2":0.0,"strategy":"Frozen Base"},
    "A2":{"initializer":best_initializer,"optimizer":"Adam","head_lr":float(selected_config["lr"]),"lr":1e-5,"fine_lr":1e-5,"batch_size":64,"dropout":0.5,"batch_norm":False,"l2":0.0,"strategy":"Partial Unfreezing","last_blocks":4}
}
additional_cv={"A1":cv_head_config(additional["A1"]),"A2":cv_partial_config(additional["A2"])}
a1_metrics,_,_=eval_config_on_test(additional["A1"])
a2_metrics,_,_,_,_=train_partial_and_test(additional["A2"],1e-5,PROBE_EPOCHS,FINE_TUNE_EPOCHS)
additional_test={"A1":a1_metrics,"A2":a2_metrics}

overall_rows=[]
for name in ["Baseline","Best Initialization","Best Regularization","Best Optimizer","Best Hyperparameters"]:
    cfg=representatives[name]
    cv=cv_results.get({"Baseline":"C1 Baseline","Best Initialization":"C2 Best Initialization","Best Regularization":"C3 Best Regularization","Best Hyperparameters":"C4 Best Hyperparameters"}.get(name,""))
    if cv is None:
        cv=cv_head_config(cfg)
    tm=representative_test[name]
    overall_rows.append([name,cv["mean"],cv["sd"],tm["accuracy"],tm["training_time_s"]])
overall_rows.append(["Fine-Tuned Model",fine_tuned_cv["mean"],fine_tuned_cv["sd"],final_metrics["accuracy"],final_metrics["training_time_s"]])
overall_table=pd.DataFrame(overall_rows,columns=["Configuration","CV Accuracy","SD","Test Accuracy","Training Time (s)"])


In [ ]:
optimizer_table.to_csv(OUT/"optimizer_comparison.csv",index=False)
cv_table.to_csv(OUT/"cross_validation_results.csv",index=False)
overall_table.to_csv(OUT/"overall_results.csv",index=False)
pd.DataFrame(report).T.to_csv(OUT/"classification_report.csv")
pd.DataFrame(cm,index=class_names,columns=class_names).to_csv(OUT/"confusion_matrix.csv")
additional_table=pd.DataFrame([[name,*r["folds"],r["mean"],r["sd"],r["time_s"],additional_test[name]["accuracy"],additional[name].get("fine_lr",additional[name]["lr"]),additional[name]["batch_size"],additional[name]["dropout"],additional[name]["strategy"]] for name,r in additional_cv.items()],columns=["Configuration","F1","F2","F3","F4","F5","Mean","SD","CV Time (s)","Test Accuracy","Learning Rate","Batch Size","Dropout","Strategy"])
additional_table.to_csv(OUT/"additional_exercise.csv",index=False)
results={
    "seed":SEED,
    "device":str(DEVICE),
    "dataset":{"trainval":len(trainval),"training":len(train_idx),"validation":len(val_idx),"test":len(testset),"classes":NUM_CLASSES,"image_size":[224,224,3]},
    "batch_normalization":{"input":bn_x.tolist(),"mean":bn_mean,"variance":bn_var,"std":bn_std,"normalized":bn_normalized.tolist()},
    "convolution_dimensions":conv_examples.to_dict(orient="records"),
    "best_initializer":best_initializer,
    "best_regularization":best_regularization,
    "best_optimizer":best_optimizer,
    "best_learning_rate":float(best_lr),
    "best_batch_size":int(best_batch),
    "best_dropout":float(best_dropout),
    "fine_tuning_learning_rates":{str(k):{"best_validation_accuracy":float(max(v["val_acc"])),"history":v} for k,v in fine_tune_lr_histories.items()},
    "best_fine_tuning_learning_rate":float(best_fine_tune_lr),
    "optimizer_table":optimizer_table.to_dict(orient="records"),
    "learning_rate_study":{str(k):v for k,v in lr_results.items()},
    "batch_size_study":{str(k):v for k,v in bs_results.items()},
    "dropout_study":{str(k):v for k,v in drop_results.items()},
    "cv":cv_results,
    "fine_tuned_cv":fine_tuned_cv,
    "selected_configuration":selected_name,
    "final_metrics":final_metrics,
    "classification_report":report,
    "overall_results":overall_table.to_dict(orient="records"),
    "additional_exercise":additional_table.to_dict(orient="records")
}
(OUT/"results.json").write_text(json.dumps(results,indent=2,allow_nan=False))
print(json.dumps({"selected_configuration":selected_name,"final_metrics":final_metrics,"outputs":str(OUT)},indent=2))
